# 02 · Análisis Exploratorio de Datos (EDA) — SOL-USD
**Diplomado en Ciencia de Datos · ENES UNAM León**

Objetivo: entender la distribución, tendencias, volatilidad y estructura temporal del precio de Solana antes de construir modelos.

> 💡 **Nota para el informe técnico:** Este notebook alimenta directamente la sección *6. Metodología* — aquí justificamos decisiones de preprocesamiento y feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

# Cargar datos
df = pd.read_csv('../data/raw/sol_usd_raw.csv', index_col=0, parse_dates=True)

# Aplanar MultiIndex en columnas si yfinance lo generó
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Datos cargados: {df.shape} | {df.index.min().date()} → {df.index.max().date()}')
df.head(3)

## 2.1 Distribución de precios

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma del precio de cierre
axes[0].hist(df['Close'], bins=40, color='#9945FF', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Close'].mean(), color='red', linestyle='--', label=f"Media: ${df['Close'].mean():.1f}")
axes[0].axvline(df['Close'].median(), color='orange', linestyle='--', label=f"Mediana: ${df['Close'].median():.1f}")
axes[0].set_title('Distribución del precio de cierre')
axes[0].set_xlabel('USD')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['Close'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#9945FF', alpha=0.6))
axes[1].set_title('Boxplot precio de cierre')
axes[1].set_ylabel('USD')

plt.tight_layout()
plt.savefig('../data/raw/02_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.2 Retornos diarios y volatilidad

Los modelos de ML trabajan mejor con **retornos** que con precios absolutos, ya que los precios tienen tendencia (no son estacionarios).

In [ ]:
# Retorno porcentual diario
df['daily_return'] = df['Close'].pct_change() * 100

# Volatilidad móvil (rolling std de 21 días ~ 1 mes)
df['volatility_21d'] = df['daily_return'].rolling(21).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Retornos diarios
colors = ['#14F195' if r >= 0 else '#FF4444' for r in df['daily_return'].fillna(0)]
axes[0].bar(df.index, df['daily_return'], color=colors, alpha=0.8)
axes[0].axhline(0, color='white', linewidth=0.8)
axes[0].set_title('Retorno diario (%) — SOL-USD')
axes[0].set_ylabel('%')

# Volatilidad rolling
axes[1].plot(df.index, df['volatility_21d'], color='#FFB800', linewidth=1.5)
axes[1].fill_between(df.index, df['volatility_21d'], alpha=0.2, color='#FFB800')
axes[1].set_title('Volatilidad móvil 21 días (desv. estándar de retornos)')
axes[1].set_ylabel('Desv. estándar (%)')

plt.tight_layout()
plt.savefig('../data/raw/02_retornos_volatilidad.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Retorno promedio diario : {df['daily_return'].mean():.3f}%")
print(f"Volatilidad promedio    : {df['daily_return'].std():.3f}%")
print(f"Peor día                : {df['daily_return'].min():.2f}%")
print(f"Mejor día               : {df['daily_return'].max():.2f}%")

## 2.3 Correlación entre variables OHLCV

In [ ]:
corr_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
corr = df[corr_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matriz de correlación — Variables OHLCV')
plt.tight_layout()
plt.savefig('../data/raw/02_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Prueba de estacionariedad (ADF Test)

Las series temporales deben ser **estacionarias** para muchos modelos. Verificamos con el test de Dickey-Fuller Aumentado.

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    estacionaria = result[1] < 0.05
    print(f"\n{'='*50}")
    print(f"Serie: {name}")
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    print(f"  ¿Estacionaria?: {'✅ SÍ' if estacionaria else '❌ NO — requiere diferenciación'}")
    return estacionaria

adf_test(df['Close'], 'Close (precio absoluto)')
adf_test(df['daily_return'], 'Retorno diario (%)')

## 2.5 Autocorrelación (ACF / PACF)

Nos dice cuántos lags del pasado tienen información predictiva — esto guía cuántos lags usar como features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(df['Close'].dropna(), lags=30, ax=axes[0], color='#9945FF')
axes[0].set_title('ACF — Precio de Cierre')

plot_pacf(df['Close'].dropna(), lags=30, ax=axes[1], color='#14F195', method='ywm')
axes[1].set_title('PACF — Precio de Cierre')

plt.tight_layout()
plt.savefig('../data/raw/02_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Interpretación:")
print("  - ACF: correlación total entre el precio y sus lags")
print("  - PACF: correlación directa (eliminando efectos intermedios)")
print("  - Barras que superan la banda azul = lags significativos → usar como features")

## 2.6 Análisis por mes

In [ ]:
df['month'] = df.index.month
df['month_name'] = df.index.strftime('%b %Y')

monthly_close = df.groupby(df.index.to_period('M'))['Close'].agg(['mean', 'min', 'max'])

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(monthly_close))
ax.fill_between(x, monthly_close['min'], monthly_close['max'], alpha=0.25, color='#9945FF', label='Rango min-max')
ax.plot(x, monthly_close['mean'], color='#9945FF', linewidth=2, marker='o', markersize=5, label='Promedio mensual')
ax.set_xticks(x)
ax.set_xticklabels([str(p) for p in monthly_close.index], rotation=45, ha='right')
ax.set_title('Precio mensual — SOL-USD (media, min, max)')
ax.set_ylabel('USD')
ax.legend()
plt.tight_layout()
plt.savefig('../data/raw/02_mensual.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Guardar dataset con columnas de EDA

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Guardamos con las columnas de retorno y volatilidad para usar en siguiente notebook
df.drop(columns=['month', 'month_name'], inplace=True, errors='ignore')
df.to_csv('../data/processed/sol_usd_eda.csv')
print('Dataset con features EDA guardado ✓')
print(df.columns.tolist())

---
## ✅ Conclusiones del EDA

| Hallazgo | Implicación para el modelo |
|---|---|
| Precio NO estacionario | Usar retornos o diferencias como features |
| Retornos SÍ estacionarios | Podemos usarlos directamente |
| Alta autocorrelación en lags 1-5 | Incluir lags 1,2,3,5,7 como features |
| Volatilidad variable (clusters) | Agregar feature de volatilidad rolling |
| Correlación alta entre O/H/L/C | Normal en OHLCV, no hay multicolinealidad problemática |

**Siguiente paso →** `03_feature_engineering.ipynb`